# Lab 05: B+ Tree Range Scans & SQLite Query Plan Benchmarking

**Z2004 Database Management Systems (BS Core) · IIT Madras Zanzibar**

Sorted leaf-linked B+ tree range queries and EXPLAIN QUERY PLAN index speedup measurement.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z2004/labs/DB05_BPlus_Tree_and_Query_Plans.ipynb)

Run cells top to bottom (`Runtime > Run all`). Built on Python `sqlite3` with full self-checks.

## 1. B+ Tree Leaf-Linked Range Scans & `EXPLAIN QUERY PLAN`

Creating a B-tree index on a selective filter column shifts SQLite's plan from full-table `SCAN` ($O(N)$) to `SEARCH USING INDEX` ($O(\log N + k)$).

In [1]:
import random
import sqlite3
import time

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE Logs (id INTEGER PRIMARY KEY, crop TEXT, yield REAL)")
rng = random.Random(2004)
crops = ["Clove", "Cashew", "Rice", "Coconut", "Cassava", "Cardamom", "Vanilla", "Pepper"]
conn.executemany("INSERT INTO Logs VALUES (?, ?, ?)", [
    (i, rng.choice(crops), rng.uniform(0.5, 8.0)) for i in range(30_000)
])

print("Before index:", conn.execute("EXPLAIN QUERY PLAN SELECT * FROM Logs WHERE crop='Vanilla'").fetchall())
t0 = time.perf_counter()
for _ in range(50):
    conn.execute("SELECT COUNT(*) FROM Logs WHERE crop='Vanilla'").fetchone()
t_scan = time.perf_counter() - t0

conn.execute("CREATE INDEX idx_crop ON Logs(crop)")
print("After index :", conn.execute("EXPLAIN QUERY PLAN SELECT * FROM Logs WHERE crop='Vanilla'").fetchall())
t0 = time.perf_counter()
for _ in range(50):
    conn.execute("SELECT COUNT(*) FROM Logs WHERE crop='Vanilla'").fetchone()
t_idx = time.perf_counter() - t0

print(f"50 queries -> Table scan: {t_scan*1000:.2f} ms | Indexed: {t_idx*1000:.2f} ms")
assert t_idx < t_scan


Before index: [(2, 0, 0, 'SCAN Logs')]
After index : [(3, 0, 0, 'SEARCH Logs USING INDEX idx_crop (crop=?)')]
50 queries -> Table scan: 110.54 ms | Indexed: 8.89 ms
